# Ground state of the Hubbard model with a hidden Pfaffian ViT

This notebook finds the ground state of the 2D Fermi-Hubbard model with `jVMC.nets.HiddenPfaffianViT`, a fermionic wave function that combines a pairing wave function with a vision transformer (ViT). It uses the spinful fermionic operators (see the fermions example), a symmetry projection, and a sampler restricted to a sector of fixed particle numbers.

In [1]:
from itertools import combinations
import jax.numpy as jnp
import jVMC
import jVMC.nets as nets
import jVMC.operator.discrete as op
from jVMC.sampler.base import AbstractSampler
from jVMC.stats import SampledObs
from jVMC.symmetry.lattice_symmetries import square_translation_symmetry
from jVMC.vqs import NQS

Running in single-node mode (JVMC_USE_DISTRIBUTED not set)


/Users/som30090/code/Custom_jVMC/jVMC/solver/util.py:10: JaxMgWarning: Unsupported platform darwin, only Linux is supported. Non-Linux only works for docs.
  import jaxmg
/Users/som30090/code/Custom_jVMC/jVMC/solver/util.py:10: JaxMgWarning: No GPUs found, only use this mode for testing or generating documentation.
  import jaxmg


## Hamiltonian

$$
H = -t \sum_{\langle i,j\rangle} \sum_{\sigma} \left( \hat c^\dagger_{i\sigma} \hat c_{j\sigma} + \mathrm{h.c.} \right) + U \sum_i \hat n_{i\uparrow} \hat n_{i\downarrow} - \mu \sum_{i,\sigma} \hat n_{i\sigma} ,
$$

on a $2 \times 2$ lattice with periodic boundary conditions, with $t = 1$, $U = 4$ and $\mu = U/2$. Each site has local dimension 4 (empty, $\uparrow$, $\downarrow$, $\uparrow\downarrow$), and the site $(x, y)$ has index `y * Lx + x`. As in the ViT example, the bonds are collected in a set: on a $2 \times 2$ lattice the bond across the boundary connects the same sites as the direct one, so there are 4 distinct bonds.

We work at half filling with $N_\uparrow = N_\downarrow = 2$. Since the particle numbers are fixed, the chemical potential only shifts the energy by $-\mu N = -8$.

In [2]:
Lx, Ly = 2, 2
L = Lx * Ly
t = 1.0
U = 4.0
mu = U / 2.0

def site(x, y):
    return y * Lx + x

bonds = set()
for y in range(Ly):
    for x in range(Lx):
        i = site(x, y)
        bonds.add(tuple(sorted((i, site((x + 1) % Lx, y)))))
        bonds.add(tuple(sorted((i, site(x, (y + 1) % Ly)))))

hamiltonian = 0
for i, j in sorted(bonds):
    for spin in ("up", "down"):
        hamiltonian += -t * (op.Creation(i, spin) * op.Annihilation(j, spin))
        hamiltonian += -t * (op.Creation(j, spin) * op.Annihilation(i, spin))
for i in range(L):
    hamiltonian += U * op.Number(i, "up") * op.Number(i, "down")
    hamiltonian += -mu * (op.Number(i, "up") + op.Number(i, "down"))

## Exact sampler in a fixed particle-number sector

$H$ conserves $N_\uparrow$ and $N_\downarrow$, and the wave function below is nonzero only for $N_\uparrow = N_\downarrow = 2$. The library's `ExactSampler` would sum over all $4^4 = 256$ configurations, including those where the wave function vanishes, so we write a small custom sampler that enumerates only the $\binom{4}{2}^2 = 36$ configurations of the sector.

A sampler derives from `jVMC.sampler.base.AbstractSampler` and implements:

- `sample()`, which stores and returns the configurations, their $\ln\psi$, and their normalised weights, here $|\psi(s)|^2 / \sum_{s'} |\psi(s')|^2$,
- `__call__(observable)`, which evaluates the local estimator of an observable on the stored samples.

In [3]:
def spinful_sector_basis(L, n_up, n_down, sample_shape):
    states = []
    for up_sites in combinations(range(L), n_up):
        up_sites = set(up_sites)
        for down_sites in combinations(range(L), n_down):
            down_sites = set(down_sites)
            state = []
            for site_idx in range(L):
                local_state = int(site_idx in up_sites) + 2 * int(site_idx in down_sites)
                state.append(local_state)
            states.append(state)
    return jnp.asarray(states, dtype=jnp.int32).reshape((-1, *sample_shape))


class FixedSpinfulSectorSampler(AbstractSampler):
    def __init__(self, psi, L, n_up, n_down):
        super().__init__(psi)
        self.basis = spinful_sector_basis(L, n_up, n_down, psi.sample_shape)
        self.num_samples = self.basis.shape[0]

    def __call__(self, observable, **obs_kwargs):
        raw_data = observable.get_O_loc(self.samples, self.psi, log_psi_s=self.log_psi, **obs_kwargs)
        return SampledObs(raw_data, self.weights)

    def sample(self, num_samples=None):
        del num_samples
        log_psi = self.psi(self.basis)
        shift = jnp.max(jnp.real(log_psi))
        # |psi|^2, shifted by the largest amplitude for numerical stability
        weights = jnp.exp((jnp.real(log_psi) - shift) / 0.5)
        weights = weights / jnp.sum(weights)
        self._samples = self.basis
        self._log_psi = log_psi
        self._weights = weights
        return self._samples, self._log_psi, self._weights

## Hidden Pfaffian ViT

The wave function is a sum over the lattice translations $g$,

$$
\psi(s) = \frac{1}{n_g} \sum_{g} w_g(s)\, e^{\mathrm{ViT}(g s)}\, \det F_{g s},
$$

where:

- $\det F_{s}$ is a singlet pairing (Pfaffian) wave function: $F$ is a trainable $L \times L$ complex matrix, and $F_s$ is its submatrix with the rows of the sites occupied by $\uparrow$ electrons and the columns of the sites occupied by $\downarrow$ electrons. `particles_per_spin` sets $N_\uparrow = N_\downarrow$; outside this sector the wave function vanishes.
- $\mathrm{ViT}(s)$ is a vision transformer that gives a configuration-dependent coefficient to each term; `patch_size`, `layers`, `embed_dim` and `heads` are its hyperparameters, as in the ViT example.
- The sum over translations, with the weights $w_g(s)$ that include the fermionic signs of the permutation, projects the state onto zero momentum. The symmetry is created by `square_translation_symmetry` with the particle type `"spinful_fermion"`.

In [4]:
symmetry = square_translation_symmetry(Lx, Ly, "spinful_fermion")
net = nets.HiddenPfaffianViT(
    Lx=Lx,
    Ly=Ly,
    symmetry=symmetry,
    patch_size=1,
    layers=2,
    embed_dim=16,
    heads=2,
    particles_per_spin=2,
    param_dtype=jnp.float64,
    compute_dtype=jnp.float64,
)
psi = NQS(net, (Ly, Lx), batch_size=64, seed=1234)
exact_sampler = FixedSpinfulSectorSampler(psi, L, n_up=2, n_down=2)
print("Number of parameters:", psi.num_parameters)

Number of parameters: 5904


## Optimisation with MinSR

We minimise the energy with `MinSR` (see the ViT example), which computes the stochastic reconfiguration update by solving a linear system whose size is the number of samples (here 36) instead of the number of parameters (here 5904). `diagonal_shift` regularises the system, `Pinv` solves it with a pseudo-inverse, and the `Euler` stepper applies the update with a constant learning rate.

The optimisation runs in blocks of 100 steps. After each block we resample (the sampler keeps the wave function of the last `sample()` call, and the optimiser updates the parameters after its last measurement) and measure the energy and its variance. The variance of the local energy vanishes for an eigenstate, so we stop once it falls below `target_variance`, or after `max_blocks` blocks.

In [5]:
loss_function = jVMC.objective_function.Observable(hamiltonian)
stepper = jVMC.stepper.Euler(5e-2)
solver = jVMC.solver.Pinv(pinv_cutoff=1e-8)
optimizer = jVMC.optimizer.MinSR(exact_sampler, psi, solver=solver, diagonal_shift=1e-3)

target_variance = 1e-8
block_steps = 100
max_blocks = 10

for block in range(max_blocks):
    optimizer.ground_state_search(block_steps, loss_function, stepper)
    exact_sampler.sample()
    final = exact_sampler(hamiltonian)
    variance = float(jnp.real(final.var[0]))
    energy = complex(final.mean[0])
    print(f"block {block + 1:02d}: E={energy.real:.12f}, Var={variance:.3e}")
    if variance < target_variance:
        break

print("Final local-energy estimate:", final)

100%|██████████| 100/100 [00:04<00:00, 23.62it/s, E=-1.0103e+01+4.7857e-20j ± 2.2643e-03 (Var = 3.7925e-05)]

Recorded timings:
    - sampling: 0.000329s
    - compute objective function and gradient: 0.005027s
    - solve: 0.004122s


block 01: E=-10.102715519774, Var=3.410e-05


100%|██████████| 100/100 [00:01<00:00, 60.75it/s, E=-1.0103e+01-4.9962e-22j ± 1.1177e-05 (Var = 9.2405e-10)]

Recorded timings:
    - sampling: 0.000336s
    - compute objective function and gradient: 0.005825s
    - solve: 0.003936s
block 02: E=-10.102748482659, Var=8.309e-10
Final local-energy estimate: -1.0103e+01-3.8712e-22j ± 1.0599e-05 (Var = 8.3093e-10)


## Comparison with exact diagonalisation

The exact ground state energy in the sector $N_\uparrow = N_\downarrow = 2$, from the diagonalisation of the $36 \times 36$ Hamiltonian matrix, is $E_0 = -10.102748483462$.

In [6]:
exact_energy = -10.102748483462  # exact diagonalisation of the 36-dimensional sector

print(f"VMC energy:     {energy.real:.12f}")
print(f"Exact energy:   {exact_energy:.12f}")
print(f"Relative error: {abs((energy.real - exact_energy) / exact_energy):.1e}")

VMC energy:     -10.102748482659
Exact energy:   -10.102748483462
Relative error: 7.9e-11
